# Angular Interview Q&A — Set 2
### Advanced Topics | Instructor Reference Guide

---

## About This Set

**Set 2** builds on Set 1 fundamentals and covers **advanced Angular topics** asked in mid-to-senior level interviews.

| Category | Questions | Topics |
|---|---|---|
| **1. Angular Signals — Deep Dive** | Q1 – Q5 | signal/computed/effect internals, toSignal/toObservable, input/output, linkedSignal, Signals vs RxJS |
| **2. State Management** | Q6 – Q10 | NgRx Store/Actions/Reducers/Effects/Selectors, NgRx Signals Store, alternatives |
| **3. Testing** | Q11 – Q15 | TestBed, component tests, service tests, HttpTesting, mocking, routing tests |
| **4. SSR & Hydration** | Q16 – Q20 | @angular/ssr setup, Non-Destructive Hydration, Incremental Hydration, Route-level Render Mode |
| **5. Advanced Concepts** | Q21 – Q25 | Angular CDK, content projection, HostBinding/Listener, InjectionToken, forRoot/forChild |
| **6. Best Practices & Architecture** | Q26 – Q30 | Smart/Dumb pattern, feature structure, environments, security (OWASP), tree-shaking |

**Total Questions: 30**

---

> **Prerequisite:** Complete Set 1 (core fundamentals) before this set.
> **Difficulty level:** Mid-level → Senior
> Look for **"Senior Trap:"** sections — questions senior candidates commonly get wrong.

# Category 1 — Angular Signals Deep Dive

---

## Q1. How does the Angular Signal graph work internally? What makes `computed()` lazy and glitch-free?

**Answer:**
The Angular signal system is built around a **reactive graph** — a directed acyclic graph (DAG) where source signals are nodes and `computed()` values are derived nodes tracking their dependencies.

**Key internal mechanics:**

**1. Dependency tracking (automatic):**
When a `computed()` or `effect()` runs, Angular records every `signal()` that is **read** (called as a function `signal()`). These become the dependencies.

```typescript
const a = signal(1);
const b = signal(2);
const c = computed(() => {
  // Angular tracks: c depends on a and b
  return a() + b();
});
```

**2. Laziness:**
`computed()` values are NOT recalculated when a dependency changes. Instead, they are marked **STALE**. Recalculation happens only when the computed is **read** again.

```typescript
a.set(10);         // c is marked STALE — NOT recalculated yet
console.log(c());  // NOW c recalculates: 10 + 2 = 12
```

**3. Glitch-free (no diamond problem):**
If both `a` and `b` change synchronously, Angular ensures `c` recalculates only **once** — not once per dependency change.

```typescript
const c = computed(() => a() + b());
const d = computed(() => c() * 2);

// Both change synchronously
a.set(5);
b.set(5);
// d recalculates once: (5+5)*2 = 20 — NOT intermediate (5+2)*2=14 then (5+5)*2=20
```

**4. Version tracking:**
Each signal has an internal **version counter**. When set, the version increments. Derived signals check the version of dependencies — if unchanged, the cached value is returned without recalculation.

**5. `effect()` scheduling:**
Effects are scheduled as **microtasks** (not synchronous) — they run after the current synchronous code completes, ensuring the signal graph is settled before side effects run.

> **Senior Trap:** "Does changing a signal immediately re-render the component?"
> No. Angular batches signal changes and schedules a re-render as a microtask. Multiple synchronous signal changes in one event handler result in a single re-render.

---

## Q2. What are `toSignal()` and `toObservable()`? When do you use each?

**Answer:**
These are **bridge functions** from `@angular/core/rxjs-interop` that allow Signals and Observables to interoperate.

**`toSignal(observable)` — Observable → Signal:**
Subscribes to an Observable and exposes its latest value as a Signal. Automatically unsubscribes when the injection context is destroyed.

```typescript
import { toSignal } from '@angular/core/rxjs-interop';
import { inject } from '@angular/core';

@Component({
  standalone: true,
  template: `
    <!-- No async pipe needed — signal is synchronous -->
    @if (user()) {
      <h1>{{ user()!.name }}</h1>
    }
    @for (product of products(); track product.id) {
      <app-product-card [product]="product" />
    }
  `
})
export class DashboardComponent {
  private userService    = inject(UserService);
  private productService = inject(ProductService);

  // Observable → Signal (requires injection context)
  user     = toSignal(this.userService.getCurrentUser());
  // With initial value (avoids undefined on first render)
  products = toSignal(this.productService.getAll(), { initialValue: [] as Product[] });
  // requireSync: true — Observable must emit synchronously (e.g., BehaviorSubject)
  theme    = toSignal(this.themeService.theme$, { requireSync: true });
}
```

**`toObservable(signal)` — Signal → Observable:**
Converts a Signal into an Observable. Emits whenever the signal changes. Useful for composing signal changes with RxJS operators.

```typescript
import { toObservable } from '@angular/core/rxjs-interop';
import { signal } from '@angular/core';
import { switchMap, debounceTime } from 'rxjs/operators';

@Component({ ... })
export class SearchComponent {
  searchQuery = signal('');

  // Signal → Observable → RxJS pipeline → Signal
  results = toSignal(
    toObservable(this.searchQuery).pipe(
      debounceTime(300),
      filter(q => q.length >= 2),
      switchMap(q => this.searchService.search(q)),
      catchError(() => of([]))
    ),
    { initialValue: [] as SearchResult[] }
  );

  onInput(event: Event) {
    this.searchQuery.set((event.target as HTMLInputElement).value);
  }
}
```

**When to use each:**

| Function | Direction | Use When |
|---|---|---|
| `toSignal()` | Observable → Signal | Using HTTP/WebSocket data in templates without `async` pipe |
| `toObservable()` | Signal → Observable | Need RxJS operators (`debounceTime`, `switchMap`) on signal changes |

> **Follow-up:** Does `toSignal()` need to be in an injection context?
> Yes — by default. To use it outside an injection context (e.g., inside a method), pass `{ injector }` as the second argument.

---

## Q3. What are Signal Inputs (`input()`) and Signal Outputs (`output()`)? How do they replace `@Input`/`@Output`?

**Answer:**
Signal inputs and outputs are the **modern, function-based replacement** for the `@Input()` and `@Output()` decorators, introduced in Angular 17 (Developer Preview) and stable in Angular 18.

**Signal Inputs — `input()`:**

```typescript
import { Component, input, computed } from '@angular/core';

@Component({
  selector: 'app-product-card',
  standalone: true,
  template: `
    <div class="card" [class.featured]="isFeatured()">
      <h3>{{ title() }}</h3>
      <p>{{ formattedPrice() }}</p>
      <span *ngIf="discount()">Save {{ discount() }}%</span>
    </div>
  `
})
export class ProductCardComponent {
  // Required input — parent MUST provide it (compile-time + runtime error if missing)
  title     = input.required<string>();
  price     = input.required<number>();

  // Optional input with default value
  isFeatured = input(false);
  discount   = input<number | null>(null);

  // Input with alias (public name differs from property name)
  productSku = input.required<string>({ alias: 'sku' });

  // Input with transform — Angular 16+ built-in transforms
  isActive  = input(false, { transform: booleanAttribute });
  maxItems  = input(10,    { transform: numberAttribute });

  // Computed values derived from inputs — reactive automatically
  formattedPrice = computed(() => `$${this.price().toFixed(2)}`);
  discountedPrice = computed(() => {
    const d = this.discount();
    return d ? this.price() * (1 - d / 100) : this.price();
  });
}
```

**Signal Outputs — `output()`:**

```typescript
import { Component, input, output } from '@angular/core';

@Component({ selector: 'app-rating', standalone: true, template: `...` })
export class RatingComponent {
  maxStars = input(5);

  // Signal output — returns OutputEmitterRef<T>
  ratingChange    = output<number>();
  ratingSubmitted = output<{ rating: number; comment: string }>();

  // Alias
  valueChange = output<number>({ alias: 'ngModelChange' });

  selectRating(star: number) {
    this.ratingChange.emit(star);
  }

  submit(rating: number, comment: string) {
    this.ratingSubmitted.emit({ rating, comment });
  }
}
```

**Comparison table:**

| Feature | `@Input()` / `@Output()` | `input()` / `output()` |
|---|---|---|
| Type | Decorator | Function |
| Return type | `T \| undefined` | `InputSignal<T>` |
| Required enforcement | `@Input({ required: true })` | `input.required<T>()` |
| Reactive | Only via `ngOnChanges` | Built-in — use in `computed()` / `effect()` |
| Transforms | Manual | `{ transform: numberAttribute }` |
| Output type | `EventEmitter<T>` | `OutputEmitterRef<T>` |
| Migration schematic | — | `ng generate @angular/core:signal-input-migration` |

---

## Q4. What is `linkedSignal()`? What is the `resource()` API?

**Answer:**

### `linkedSignal()` — Writable Derived Signal

A `linkedSignal` is a **writable signal whose default value is derived from another signal**, but which can be manually overridden. When the source signal changes, the linked signal resets to the new computed value.

```typescript
import { signal, linkedSignal } from '@angular/core';

const pages = signal([1, 2, 3, 4, 5]);

// currentPage follows pages[0] by default but user can override
const currentPage = linkedSignal(() => pages()[0]);

console.log(currentPage()); // 1 (derived)
currentPage.set(3);         // manual override
console.log(currentPage()); // 3

pages.set([10, 11, 12]);    // source changes → resets
console.log(currentPage()); // 10 (reset to derived value)
```

**Advanced form with `previous` value:**
```typescript
const selectedTab = linkedSignal<string, string>({
  source: () => this.availableTabs()[0],
  computation: (newSource, previous) => {
    // Keep previous selection if it still exists in new tabs
    const prevValue = previous?.value;
    return this.availableTabs().includes(prevValue ?? '')
      ? prevValue!
      : newSource;
  }
});
```

### `resource()` API (Angular 19+) — Async Signal-based Data Fetching

`resource()` wraps an async operation (like an HTTP call) into a signal-aware container, providing `value`, `status`, `error`, and `isLoading` signals.

```typescript
import { resource, signal, inject } from '@angular/core';
import { HttpClient } from '@angular/common/http';
import { firstValueFrom } from 'rxjs';

@Component({
  selector: 'app-user-detail',
  template: `
    @if (userResource.isLoading()) {
      <p>Loading...</p>
    } @else if (userResource.error()) {
      <p class="error">{{ userResource.error() }}</p>
    } @else {
      <h1>{{ userResource.value()?.name }}</h1>
    }
    <button (click)="userId.set(userId() + 1)">Next User</button>
  `
})
export class UserDetailComponent {
  private http = inject(HttpClient);
  userId = signal(1);

  // Re-fetches automatically when userId() changes
  userResource = resource({
    request: () => ({ id: this.userId() }),
    loader: ({ request }) =>
      firstValueFrom(this.http.get<User>(`/api/users/${request.id}`))
  });
}
```

**`httpResource()` — HTTP-specific shorthand (Angular 19+):**
```typescript
import { httpResource } from '@angular/core/http';

userResource = httpResource<User>(() => `/api/users/${this.userId()}`);
// Automatically re-fetches when userId() changes
// Provides: .value(), .isLoading(), .error(), .status()
```

---

## Q5. Signals vs RxJS Observables — when to use each?

**Answer:**
Signals and Observables are **complementary** — not competing. They solve different problems.

**Use Signals for:**
- **Synchronous reactive state** — UI state, form fields, toggle flags
- **Derived/computed values** — calculated from other state
- **Component-local state** — values that drive the template
- **Shared application state** — in a signal-based service
- **Replacing `BehaviorSubject`** for state holding

```typescript
// Signals — synchronous, simple, great for state
const count      = signal(0);
const isLoggedIn = signal(false);
const cartItems  = signal<CartItem[]>([]);
const cartTotal  = computed(() => cartItems().reduce((s, i) => s + i.price, 0));
```

**Use RxJS Observables for:**
- **Asynchronous operations** — HTTP, WebSockets, timers
- **Complex async flows** — `switchMap`, `mergeMap`, `debounceTime`, retry logic
- **Event streams** — user events, route changes, form `valueChanges`
- **Multi-step pipelines** — transform, filter, combine multiple streams

```typescript
// RxJS — async, pipeline-based
searchResults$ = this.searchInput.valueChanges.pipe(
  debounceTime(300),
  distinctUntilChanged(),
  switchMap(q => this.http.get(`/search?q=${q}`))
);
```

**Bridge between them:**
```typescript
// HTTP Observable → Signal (for template consumption)
users = toSignal(this.http.get<User[]>('/api/users'), { initialValue: [] });

// Signal → Observable (to use RxJS operators on signal changes)
searchResults$ = toObservable(this.searchQuery).pipe(
  debounceTime(300),
  switchMap(q => this.http.get('/search', { params: { q } }))
);
results = toSignal(searchResults$, { initialValue: [] });
```

**Decision matrix:**

| Scenario | Use |
|---|---|
| Toggle button state | `signal(false)` |
| Search box with API call | `toObservable(signal).pipe(debounceTime, switchMap)` |
| HTTP GET request | `HttpClient` Observable → `toSignal()` |
| Form input validation | Reactive Forms `valueChanges` Observable |
| Shopping cart total | `computed(() => cartItems().reduce(...))` |
| WebSocket messages | RxJS `webSocket()` Observable |
| User preferences | `signal()` with `localStorage` in `effect()` |
| Polling interval | `interval(5000).pipe(switchMap(...))` |

> **Senior Trap:** "Can you put `signal.set()` inside an `effect()`?"
> Avoid it — it creates circular dependency risk and Angular will throw `NG0600: Writing to signals is not allowed in reactive context`. Use `untracked()` if you must, or restructure with `computed()` / `linkedSignal()` instead.

# Category 2 — State Management

---

## Q6. What is NgRx? Explain the core concepts: Store, Actions, Reducers, Selectors, Effects.

**Answer:**
**NgRx** is a reactive state management library for Angular based on the **Redux pattern**, using RxJS Observables. It provides a single, predictable, immutable state tree for the entire application.

**Core architecture:**

```
Component  ──dispatch(Action)──▶  Store
    │                               │
    │                           Reducer
    │                               │
    └──select(Selector)◀── new State
                                    │
                                Effect ──▶ API / Side Effect
```

**1. Store — single source of truth:**
```typescript
// The entire app state is one object tree
interface AppState {
  products: ProductState;
  cart:     CartState;
  auth:     AuthState;
}
```

**2. Actions — describe what happened:**
```typescript
// products.actions.ts
import { createAction, props } from '@ngrx/store';
import { Product } from './product.model';

export const loadProducts    = createAction('[Product List] Load Products');
export const loadProductsSuccess = createAction(
  '[Product API] Load Products Success',
  props<{ products: Product[] }>()
);
export const loadProductsFailure = createAction(
  '[Product API] Load Products Failure',
  props<{ error: string }>()
);
export const addToCart = createAction(
  '[Product Card] Add To Cart',
  props<{ product: Product }>()
);
```

**3. Reducers — pure functions that update state:**
```typescript
// products.reducer.ts
import { createReducer, on } from '@ngrx/store';

export interface ProductState {
  products:  Product[];
  loading:   boolean;
  error:     string | null;
}

const initialState: ProductState = {
  products: [],
  loading:  false,
  error:    null,
};

export const productReducer = createReducer(
  initialState,
  on(loadProducts,         state => ({ ...state, loading: true, error: null })),
  on(loadProductsSuccess,  (state, { products }) => ({ ...state, loading: false, products })),
  on(loadProductsFailure,  (state, { error })    => ({ ...state, loading: false, error })),
);
```

**4. Selectors — compute derived state efficiently:**
```typescript
// products.selectors.ts
import { createFeatureSelector, createSelector } from '@ngrx/store';

const selectProductState = createFeatureSelector<ProductState>('products');

export const selectAllProducts = createSelector(
  selectProductState, state => state.products
);
export const selectLoading = createSelector(
  selectProductState, state => state.loading
);
export const selectFeaturedProducts = createSelector(
  selectAllProducts, products => products.filter(p => p.isFeatured)
);
// Parameterized selector
export const selectProductById = (id: string) => createSelector(
  selectAllProducts, products => products.find(p => p.id === id)
);
```

**5. Effects — handle side effects (API calls):**
```typescript
// products.effects.ts
import { Injectable, inject } from '@angular/core';
import { Actions, createEffect, ofType } from '@ngrx/effects';
import { switchMap, map, catchError } from 'rxjs/operators';
import { of } from 'rxjs';

@Injectable()
export class ProductEffects {
  private actions$ = inject(Actions);
  private productService = inject(ProductService);

  loadProducts$ = createEffect(() =>
    this.actions$.pipe(
      ofType(loadProducts),
      switchMap(() =>
        this.productService.getAll().pipe(
          map(products => loadProductsSuccess({ products })),
          catchError(error => of(loadProductsFailure({ error: error.message })))
        )
      )
    )
  );
}
```

---

## Q7. How do you use the NgRx Store in a Component?

**Answer:**
Components interact with NgRx Store via `Store` service — dispatching actions and selecting state slices.

```typescript
// product-list.component.ts
import { Component, OnInit, inject } from '@angular/core';
import { Store } from '@ngrx/store';
import { AsyncPipe, NgFor } from '@angular/common';

@Component({
  selector: 'app-product-list',
  standalone: true,
  imports: [AsyncPipe, NgFor],
  template: `
    @if (loading$ | async) {
      <app-spinner />
    }
    @if (error$ | async; as error) {
      <p class="error">{{ error }}</p>
    }
    @for (product of products$ | async; track product.id) {
      <app-product-card
        [product]="product"
        (addToCart)="onAddToCart(product)" />
    }
  `
})
export class ProductListComponent implements OnInit {
  private store = inject(Store);

  // Select slices of state as Observables
  products$ = this.store.select(selectAllProducts);
  loading$  = this.store.select(selectLoading);
  error$    = this.store.select(selectError);

  ngOnInit() {
    // Dispatch action to trigger Effect → API call
    this.store.dispatch(loadProducts());
  }

  onAddToCart(product: Product) {
    this.store.dispatch(addToCart({ product }));
  }
}
```

**NgRx Store setup (standalone app):**
```typescript
// app.config.ts
import { provideStore } from '@ngrx/store';
import { provideEffects } from '@ngrx/effects';
import { provideStoreDevtools } from '@ngrx/store-devtools';

export const appConfig: ApplicationConfig = {
  providers: [
    provideStore({ products: productReducer, cart: cartReducer }),
    provideEffects([ProductEffects, CartEffects]),
    provideStoreDevtools({ maxAge: 25, logOnly: !isDevMode() }),
  ]
};
```

---

## Q8. What is NgRx Signals Store (`@ngrx/signals`)?

**Answer:**
**NgRx Signals Store** is a lightweight, signal-based state management solution released in NgRx 17. It replaces the traditional Store/Actions/Reducers/Effects pattern with a simpler, composable API built on Angular Signals.

```typescript
// product.store.ts
import { signalStore, withState, withComputed, withMethods, patchState } from '@ngrx/signals';
import { withEntities, setAllEntities } from '@ngrx/signals/entities';
import { computed, inject } from '@angular/core';
import { rxMethod } from '@ngrx/signals/rxjs-interop';
import { pipe, switchMap, tap } from 'rxjs';
import { tapResponse } from '@ngrx/operators';

export const ProductStore = signalStore(
  { providedIn: 'root' },

  // 1. State shape
  withState({
    loading: false,
    error: null as string | null,
    filter: 'all' as string,
  }),

  // 2. Entity collection (normalized)
  withEntities<Product>(),

  // 3. Computed (derived signals)
  withComputed(({ entities, filter }) => ({
    filteredProducts: computed(() =>
      filter() === 'all'
        ? entities()
        : entities().filter(p => p.category === filter())
    ),
    totalCount: computed(() => entities().length),
  })),

  // 4. Methods (update state + handle side effects)
  withMethods((store, productService = inject(ProductService)) => ({
    // Sync method
    setFilter(filter: string) {
      patchState(store, { filter });
    },

    // Async method using rxMethod (handles Observable side effects)
    loadProducts: rxMethod<void>(
      pipe(
        tap(() => patchState(store, { loading: true })),
        switchMap(() =>
          productService.getAll().pipe(
            tapResponse({
              next: products => patchState(store, setAllEntities(products), { loading: false }),
              error: (err: Error) => patchState(store, { error: err.message, loading: false }),
            })
          )
        )
      )
    ),
  }))
);
```

```typescript
// product-list.component.ts
@Component({
  standalone: true,
  template: `
    @if (store.loading()) { <app-spinner /> }

    <select (change)="store.setFilter($any($event.target).value)">
      <option value="all">All</option>
      <option value="electronics">Electronics</option>
    </select>

    @for (product of store.filteredProducts(); track product.id) {
      <app-product-card [product]="product" />
    }
    <p>Total: {{ store.totalCount() }}</p>
  `
})
export class ProductListComponent implements OnInit {
  store = inject(ProductStore);

  ngOnInit() { this.store.loadProducts(); }
}
```

**NgRx Store vs NgRx Signals Store:**

| Feature | NgRx Store (Redux) | NgRx Signals Store |
|---|---|---|
| Learning curve | Steep | Gentle |
| Boilerplate | High (Actions/Reducers/Effects) | Low |
| State access | Observables via `select()` | Signals directly |
| DevTools | Full Redux DevTools | Basic |
| Time-travel debugging | ✅ Yes | Limited |
| Best for | Large teams, complex flows | Most apps, simpler code |

---

## Q9. What are the alternatives to NgRx for state management in Angular?

**Answer:**

**1. Services + Signals (built-in — no library needed):**
```typescript
@Injectable({ providedIn: 'root' })
export class CartStore {
  private _items  = signal<CartItem[]>([]);
  private _coupon = signal<string | null>(null);

  // Public read-only signals
  readonly items    = this._items.asReadonly();
  readonly coupon   = this._coupon.asReadonly();
  readonly count    = computed(() => this._items().reduce((s, i) => s + i.qty, 0));
  readonly subtotal = computed(() => this._items().reduce((s, i) => s + i.price * i.qty, 0));
  readonly total    = computed(() => {
    const disc = this._coupon() ? 0.1 : 0;
    return this.subtotal() * (1 - disc);
  });

  addItem(item: CartItem)     { this._items.update(c => [...c, item]); }
  removeItem(id: string)      { this._items.update(c => c.filter(i => i.id !== id)); }
  applyCoupon(code: string)   { this._coupon.set(code); }
  clear()                     { this._items.set([]); this._coupon.set(null); }
}
```

**2. NGXS:**
```typescript
@State<CartStateModel>({ name: 'cart', defaults: { items: [] } })
@Injectable()
export class CartState {
  @Action(AddToCart)
  add({ getState, patchState }: StateContext<CartStateModel>, { item }: AddToCart) {
    patchState({ items: [...getState().items, item] });
  }
}
```

**3. Akita:**
```typescript
@StoreConfig({ name: 'products' })
export class ProductsStore extends EntityStore<ProductsState> {}
```

**Comparison:**

| Solution | Complexity | Bundle | DevTools | Best For |
|---|---|---|---|---|
| Services + Signals | Low | 0KB extra | None | Small–medium apps |
| NgRx Signals Store | Low–Med | ~15KB | Basic | Medium apps |
| NgRx (Redux) | High | ~25KB | Full Redux DevTools | Large, complex apps |
| NGXS | Medium | ~20KB | NGXS DevTools | Teams from Redux background |

---

## Q10. What is the difference between `Subject`, `BehaviorSubject`, and `ReplaySubject`? Which replaces which with Signals?

**Answer:**

| Type | Replays on subscribe | Initial value | Replaces with Signals |
|---|---|---|---|
| `Subject` | ❌ No | None | `effect()` side effects |
| `BehaviorSubject(val)` | ✅ Last value | Required | `signal(val)` |
| `ReplaySubject(n)` | ✅ Last N values | None | Partial — `signal` + buffer logic |
| `AsyncSubject` | ✅ Last on complete | None | No direct equivalent |

```typescript
// BehaviorSubject pattern (old)
private _count = new BehaviorSubject(0);
count$ = this._count.asObservable();
setCount(n: number) { this._count.next(n); }

// Signal equivalent (new)
count = signal(0);
// Use count() in templates — no async pipe needed
```

```typescript
// Subject for events (still valid — use alongside signals)
private destroy$ = new Subject<void>();

ngOnInit() {
  this.stream$.pipe(takeUntil(this.destroy$)).subscribe(...);
}
ngOnDestroy() { this.destroy$.next(); this.destroy$.complete(); }

// Modern replacement with takeUntilDestroyed (Angular 16+)
ngOnInit() {
  this.stream$.pipe(takeUntilDestroyed(this.destroyRef)).subscribe(...);
}
```

> **Senior Trap:** "Are BehaviorSubjects still needed in Angular 18+?"
> For **state holding** — no. Use `signal()` instead. For **event streams** (one-off notifications, no state) — `Subject` is still the right tool.

# Category 3 — Testing

---

## Q11. How do you unit test an Angular Component? Explain `TestBed`.

**Answer:**
**`TestBed`** is Angular's primary testing utility — it creates a mini Angular environment to test components, services, and pipes in isolation.

```typescript
// counter.component.spec.ts
import { ComponentFixture, TestBed } from '@angular/core/testing';
import { By } from '@angular/platform-browser';
import { CounterComponent } from './counter.component';

describe('CounterComponent', () => {
  let fixture: ComponentFixture<CounterComponent>;
  let component: CounterComponent;

  beforeEach(async () => {
    await TestBed.configureTestingModule({
      imports: [CounterComponent],  // Standalone component → goes in imports
      // declarations: [CounterComponent],  // NgModule-based → goes in declarations
    }).compileComponents();

    fixture   = TestBed.createComponent(CounterComponent);
    component = fixture.componentInstance;
    fixture.detectChanges();  // Triggers ngOnInit and initial rendering
  });

  // Test component class logic
  it('should create', () => {
    expect(component).toBeTruthy();
  });

  it('should start count at 0', () => {
    expect(component.count()).toBe(0);
  });

  it('should increment count on increment()', () => {
    component.increment();
    expect(component.count()).toBe(1);
  });

  // Test template/DOM
  it('should display count in template', () => {
    component.count.set(5);
    fixture.detectChanges();  // Re-render after signal change

    const countEl = fixture.debugElement.query(By.css('[data-testid="count"]'));
    expect(countEl.nativeElement.textContent).toContain('5');
  });

  // Test button click
  it('should increment when + button is clicked', () => {
    const button = fixture.debugElement.query(By.css('[data-testid="increment-btn"]'));
    button.nativeElement.click();
    fixture.detectChanges();

    expect(component.count()).toBe(1);
  });

  // Test @Input binding
  it('should display provided title', () => {
    component.title = 'My Counter';
    fixture.detectChanges();

    const h1 = fixture.debugElement.query(By.css('h1'));
    expect(h1.nativeElement.textContent).toContain('My Counter');
  });

  // Test @Output event
  it('should emit countChanged when count changes', () => {
    const emittedValues: number[] = [];
    component.countChanged.subscribe((v: number) => emittedValues.push(v));

    component.increment();
    component.increment();

    expect(emittedValues).toEqual([1, 2]);
  });
});
```

**Key `TestBed` concepts:**

| Concept | Description |
|---|---|
| `configureTestingModule` | Sets up the mini Angular module for the test |
| `createComponent` | Creates the component and returns `ComponentFixture` |
| `fixture.detectChanges()` | Triggers change detection (must call after state changes) |
| `fixture.componentInstance` | The component class instance |
| `fixture.nativeElement` | The root DOM element |
| `fixture.debugElement` | Angular's debug wrapper (use `.query(By.css(...))`) |
| `By.css(selector)` | Predicate for querying by CSS selector |

---

## Q12. How do you test an Angular Service?

**Answer:**
Services without HTTP can be tested as plain TypeScript classes. Services with HTTP use `HttpClientTestingModule`.

**Testing a pure service:**
```typescript
// auth.service.spec.ts
import { TestBed } from '@angular/core/testing';
import { AuthService } from './auth.service';

describe('AuthService', () => {
  let service: AuthService;

  beforeEach(() => {
    TestBed.configureTestingModule({});
    service = TestBed.inject(AuthService);
  });

  it('should be created', () => {
    expect(service).toBeTruthy();
  });

  it('should return false when not logged in', () => {
    expect(service.isLoggedIn()).toBeFalse();
  });

  it('should return true after login', () => {
    service.login('user@test.com', 'password');
    expect(service.isLoggedIn()).toBeTrue();
  });

  it('should clear user on logout', () => {
    service.login('user@test.com', 'password');
    service.logout();
    expect(service.isLoggedIn()).toBeFalse();
    expect(service.currentUser()).toBeNull();
  });
});
```

**Testing a service with dependencies (mocking):**
```typescript
// order.service.spec.ts
import { TestBed } from '@angular/core/testing';
import { OrderService } from './order.service';
import { AuthService } from './auth.service';

describe('OrderService', () => {
  let service: OrderService;
  let authServiceSpy: jasmine.SpyObj<AuthService>;

  beforeEach(() => {
    // Create a spy object — mock all methods
    authServiceSpy = jasmine.createSpyObj('AuthService', ['isLoggedIn', 'getToken']);
    authServiceSpy.isLoggedIn.and.returnValue(true);
    authServiceSpy.getToken.and.returnValue('mock-token-123');

    TestBed.configureTestingModule({
      providers: [
        OrderService,
        { provide: AuthService, useValue: authServiceSpy }  // Inject mock
      ]
    });
    service = TestBed.inject(OrderService);
  });

  it('should place order when logged in', () => {
    const result = service.placeOrder({ items: ['product-1'] });
    expect(authServiceSpy.isLoggedIn).toHaveBeenCalled();
    expect(result).toBeTruthy();
  });

  it('should throw error when not logged in', () => {
    authServiceSpy.isLoggedIn.and.returnValue(false);
    expect(() => service.placeOrder({ items: [] })).toThrowError('Not authenticated');
  });
});
```

---

## Q13. How do you test HTTP calls in Angular?

**Answer:**
Use `HttpClientTestingModule` and `HttpTestingController` to intercept and mock HTTP requests without making real network calls.

```typescript
// product.service.spec.ts
import { TestBed } from '@angular/core/testing';
import { HttpClientTestingModule, HttpTestingController } from '@angular/common/http/testing';
import { ProductService } from './product.service';
import { Product } from './product.model';

describe('ProductService', () => {
  let service: ProductService;
  let httpMock: HttpTestingController;

  const mockProducts: Product[] = [
    { id: '1', name: 'Laptop', price: 999 },
    { id: '2', name: 'Phone',  price: 499 },
  ];

  beforeEach(() => {
    TestBed.configureTestingModule({
      imports: [HttpClientTestingModule],
      providers: [ProductService],
    });
    service  = TestBed.inject(ProductService);
    httpMock = TestBed.inject(HttpTestingController);
  });

  afterEach(() => {
    // Verify no unexpected requests were made
    httpMock.verify();
  });

  it('should load all products', () => {
    let result: Product[] | undefined;

    service.getAll().subscribe(products => result = products);

    // Expect one GET request to the products endpoint
    const req = httpMock.expectOne('https://api.example.com/products');
    expect(req.request.method).toBe('GET');

    // Flush mock data (simulate API response)
    req.flush(mockProducts);

    expect(result).toEqual(mockProducts);
    expect(result!.length).toBe(2);
  });

  it('should load product by id', () => {
    let result: Product | undefined;

    service.getById('1').subscribe(p => result = p);

    const req = httpMock.expectOne('https://api.example.com/products/1');
    expect(req.request.method).toBe('GET');
    req.flush(mockProducts[0]);

    expect(result?.name).toBe('Laptop');
  });

  it('should handle 404 error', () => {
    let error: any;

    service.getById('999').subscribe({
      next: () => fail('should have failed'),
      error: err => error = err,
    });

    const req = httpMock.expectOne('https://api.example.com/products/999');
    req.flush('Not found', { status: 404, statusText: 'Not Found' });

    expect(error).toBeTruthy();
  });

  it('should create a product with POST', () => {
    const newProduct = { name: 'Tablet', price: 299 };
    service.create(newProduct).subscribe();

    const req = httpMock.expectOne('https://api.example.com/products');
    expect(req.request.method).toBe('POST');
    expect(req.request.body).toEqual(newProduct);
    req.flush({ id: '3', ...newProduct });
  });
});
```

---

## Q14. What is mocking in Angular tests? How do you use `spyOn` and stub services?

**Answer:**
**Mocking** replaces real dependencies with controlled test doubles, ensuring unit tests are fast, deterministic, and isolated.

**3 mocking strategies:**

**1. `jasmine.createSpyObj` — mock entire service:**
```typescript
const routerSpy = jasmine.createSpyObj('Router', ['navigate', 'navigateByUrl']);
routerSpy.navigate.and.returnValue(Promise.resolve(true));

TestBed.configureTestingModule({
  providers: [{ provide: Router, useValue: routerSpy }]
});
```

**2. `spyOn` — mock individual methods on real object:**
```typescript
it('should call navigate on successful login', () => {
  const router = TestBed.inject(Router);
  const navigateSpy = spyOn(router, 'navigate');

  component.login();

  expect(navigateSpy).toHaveBeenCalledWith(['/dashboard']);
});
```

**3. Stub class — lightweight fake implementation:**
```typescript
// Test stub for AuthService
class AuthServiceStub {
  isLoggedIn = signal(true);
  currentUser = signal({ id: '1', name: 'Test User', role: 'admin' });
  login()  { this.isLoggedIn.set(true); }
  logout() { this.isLoggedIn.set(false); }
}

TestBed.configureTestingModule({
  providers: [{ provide: AuthService, useClass: AuthServiceStub }]
});
```

**Common spy matchers:**
```typescript
expect(spy).toHaveBeenCalled();
expect(spy).toHaveBeenCalledTimes(2);
expect(spy).toHaveBeenCalledWith('/dashboard');
expect(spy).not.toHaveBeenCalled();

// Return values
spy.and.returnValue('mock value');
spy.and.returnValues('first', 'second', 'third');
spy.and.callFake((arg) => arg.toUpperCase());
spy.and.throwError(new Error('test error'));

// Async
spy.and.resolveTo({ data: [] });
spy.and.rejectWith(new Error('network error'));
```

**Testing Observables (async):**
```typescript
it('should load users', (done) => {
  const mockUsers = [{ id: 1, name: 'Alice' }];
  spyOn(userService, 'getAll').and.returnValue(of(mockUsers));

  component.loadUsers();

  component.users$.subscribe(users => {
    expect(users).toEqual(mockUsers);
    done();  // Signal async test completion
  });
});

// With fakeAsync for synchronous-style async testing
it('should load after debounce', fakeAsync(() => {
  component.search('angular');
  tick(300);  // Fast-forward 300ms debounce
  fixture.detectChanges();
  expect(component.results().length).toBeGreaterThan(0);
}));
```

---

## Q15. How do you test Angular Routes and Guards?

**Answer:**

**Testing Router navigation:**
```typescript
// app.component.spec.ts
import { TestBed } from '@angular/core/testing';
import { RouterTestingModule } from '@angular/router/testing';
import { Router } from '@angular/router';
import { Location } from '@angular/common';

describe('App Routing', () => {
  let router: Router;
  let location: Location;

  beforeEach(async () => {
    await TestBed.configureTestingModule({
      imports: [
        RouterTestingModule.withRoutes([
          { path: '',         component: HomeComponent },
          { path: 'products', component: ProductListComponent },
          { path: 'login',    component: LoginComponent },
        ])
      ],
      declarations: [HomeComponent, ProductListComponent, LoginComponent],
    }).compileComponents();

    router   = TestBed.inject(Router);
    location = TestBed.inject(Location);
    router.initialNavigation();
  });

  it('should navigate to /products', async () => {
    await router.navigate(['/products']);
    expect(location.path()).toBe('/products');
  });
});
```

**Testing a Route Guard (functional):**
```typescript
// auth.guard.spec.ts
import { TestBed } from '@angular/core/testing';
import { Router } from '@angular/router';
import { AuthService } from './auth.service';
import { authGuard } from './auth.guard';

describe('authGuard', () => {
  let authServiceSpy: jasmine.SpyObj<AuthService>;
  let routerSpy: jasmine.SpyObj<Router>;

  beforeEach(() => {
    authServiceSpy = jasmine.createSpyObj('AuthService', ['isLoggedIn']);
    routerSpy      = jasmine.createSpyObj('Router', ['createUrlTree', 'navigate']);
    routerSpy.createUrlTree.and.returnValue({ } as any);

    TestBed.configureTestingModule({
      providers: [
        { provide: AuthService, useValue: authServiceSpy },
        { provide: Router,      useValue: routerSpy },
      ]
    });
  });

  it('should allow access when logged in', () => {
    authServiceSpy.isLoggedIn.and.returnValue(true);

    const result = TestBed.runInInjectionContext(() =>
      authGuard({} as any, {} as any)
    );

    expect(result).toBeTrue();
  });

  it('should redirect to /login when not logged in', () => {
    authServiceSpy.isLoggedIn.and.returnValue(false);

    TestBed.runInInjectionContext(() => authGuard({} as any, { url: '/dashboard' } as any));

    expect(routerSpy.createUrlTree).toHaveBeenCalledWith(
      ['/login'], jasmine.objectContaining({ queryParams: { returnUrl: '/dashboard' } })
    );
  });
});

# Category 4 — SSR & Hydration

---

## Q16. What is Angular SSR? How do you set it up with `@angular/ssr`?

**Answer:**
**Server-Side Rendering (SSR)** means Angular renders the application's HTML on the **Node.js server** before sending it to the browser. The browser receives pre-rendered HTML, making the page visible immediately (fast FCP/LCP), after which Angular "takes over" by hydrating the client-side app.

**Why SSR matters:**
- **SEO** — Search engine crawlers see real HTML content
- **Performance** — First Contentful Paint and LCP improve significantly
- **Social sharing** — OpenGraph meta tags are server-rendered

**Setup:**
```bash
# Add SSR to existing project
ng add @angular/ssr

# Create new project with SSR
ng new my-app --ssr
```

**What `ng add @angular/ssr` generates:**
```
app/
  app.component.ts
  app.config.ts          ← Browser providers (CSR)
  app.config.server.ts   ← Server providers (SSR)
  app.routes.ts
  app.routes.server.ts   ← Route-level render mode config
server.ts                ← Express server entry point
```

```typescript
// app.config.server.ts
import { mergeApplicationConfig, ApplicationConfig } from '@angular/core';
import { provideServerRendering } from '@angular/platform-server';
import { provideServerRoutesConfig } from '@angular/ssr';
import { appConfig } from './app.config';
import { serverRoutes } from './app.routes.server';

const serverConfig: ApplicationConfig = {
  providers: [
    provideServerRendering(),
    provideServerRoutesConfig(serverRoutes),
  ]
};

export const config = mergeApplicationConfig(appConfig, serverConfig);
```

```typescript
// server.ts — Express server
import { APP_BASE_HREF } from '@angular/common';
import { CommonEngine } from '@angular/ssr';
import express from 'express';
import bootstrap from './src/main.server';

const app = express();
const engine = new CommonEngine();

app.get('**', (req, res, next) => {
  engine.render({
    bootstrap,
    documentFilePath: indexHtml,
    url: req.originalUrl,
    publicPath: distFolder,
    providers: [{ provide: APP_BASE_HREF, useValue: req.baseUrl }],
  })
  .then(html => res.send(html))
  .catch(err => next(err));
});
```

```bash
# Build for SSR
ng build

# Run SSR server
node dist/my-app/server/server.mjs
```

---

## Q17. What is Non-Destructive Hydration? How does it differ from old SSR?

**Answer:**

**Old SSR (pre-Angular 16) — Destructive:**
```
Server renders HTML → Browser receives HTML → User sees content (fast FCP)
                                            → Angular boots, DESTROYS server HTML
                                            → Re-renders everything from scratch
                                            → Flickering / layout shift during takeover
```

**Non-Destructive Hydration (Angular 16+ stable):**
```
Server renders HTML → Browser receives HTML → User sees content (fast FCP)
                                            → Angular boots, REUSES existing server DOM
                                            → Attaches event listeners, activates signals/Observables
                                            → No flickering — same DOM, now interactive
```

**Setup:**
```typescript
// app.config.ts
import { provideClientHydration } from '@angular/platform-browser';

export const appConfig: ApplicationConfig = {
  providers: [
    provideRouter(routes),
    provideClientHydration(),  // ← Enable hydration (stable since Angular 17)
    provideHttpClient(withFetch()),  // withFetch() required for SSR
  ]
};
```

**Why `withFetch()` matters for SSR:**
Angular SSR uses the Node.js environment where `XMLHttpRequest` doesn't exist. `withFetch()` switches `HttpClient` to the native `fetch` API, which works on both server and browser.

**HTTP Transfer State — avoiding duplicate API calls:**
```typescript
// Without transfer state: API called TWICE (once on server, once on client)
// With provideClientHydration() — Angular automatically caches server-side HTTP responses
// and replays them on the client without re-fetching

// app.config.ts
provideClientHydration(withHttpTransferCache())  // Included by default
```

**Comparison:**

| Aspect | Old SSR | Non-Destructive Hydration |
|---|---|---|
| Server DOM after boot | Destroyed | Reused |
| Flickering | Yes | No |
| HTTP calls | Duplicated | Cached via Transfer State |
| Event handling | Delayed | Immediate after hydration |

---

## Q18. What is Incremental Hydration? How does it differ from full hydration?

**Answer:**
**Incremental Hydration** (Developer Preview — Angular 18, stable direction in Angular 19) extends Non-Destructive Hydration to allow **deferring hydration of specific component subtrees** until they are needed.

**Full Hydration:** Angular hydrates the entire page at once — all components are activated, even those below the fold that the user hasn't interacted with.

**Incremental Hydration:** Only critical above-fold components are hydrated immediately. Below-fold, non-critical components are hydrated **on demand** based on triggers.

```typescript
// app.config.ts — Enable incremental hydration
import { provideClientHydration, withIncrementalHydration } from '@angular/platform-browser';

export const appConfig: ApplicationConfig = {
  providers: [
    provideClientHydration(withIncrementalHydration()),
  ]
};
```

```html
<!-- app.component.html -->

<!-- Hydrated immediately (critical, above fold) -->
<app-hero-banner />
<app-product-showcase />

<!-- Hydrated when this section enters the viewport -->
@defer (hydrate on viewport) {
  <app-feature-comparison />
}

<!-- Hydrated only when user clicks or focuses -->
@defer (hydrate on interaction) {
  <app-live-chat-widget />
} @placeholder {
  <div class="chat-placeholder">💬 Click to open chat</div>
}

<!-- Hydrated during browser idle time -->
@defer (hydrate on idle) {
  <app-analytics-tracker />
  <app-newsletter-popup />
}

<!-- Hydrated when a signal becomes true -->
@defer (hydrate when isUserLoggedIn()) {
  <app-personalized-recommendations />
}
```

**How it works internally:**
1. Server renders full HTML for all components (including deferred ones)
2. Browser displays complete HTML immediately (good FCP)
3. Angular hydrates only the critical components upfront
4. Deferred components' JavaScript is **not parsed** until triggered
5. On trigger → JS is loaded → component hydrates in place

**Performance impact example:**
```
Initial JS parsed:    Full hydration = 480KB  |  Incremental = 95KB
Time to Interactive:  Full = 5.8s             |  Incremental = 1.9s
```

---

## Q19. What is Route-level Render Mode? Explain `Prerender`, `Server`, and `Client`.

**Answer:**
Route-level Render Mode (Developer Preview — Angular 18) allows **per-route configuration** of the rendering strategy, instead of applying one strategy globally.

```typescript
// app.routes.server.ts
import { RenderMode, ServerRoute } from '@angular/ssr';

export const serverRoutes: ServerRoute[] = [

  // PRERENDER — HTML generated at BUILD TIME, served as static files
  // Use for: marketing pages, blog posts, docs, regulatory pages
  { path: '',         renderMode: RenderMode.Prerender },
  { path: 'pricing',  renderMode: RenderMode.Prerender },
  { path: 'about',    renderMode: RenderMode.Prerender },

  // Prerender parameterized routes — specify all possible param values
  { path: 'blog/:slug', renderMode: RenderMode.Prerender,
    async getPrerenderParams() {
      const slugs = await fetch('https://api.example.com/blog/slugs')
        .then(r => r.json());
      return slugs.map((slug: string) => ({ slug }));
    },
    fallback: RenderMode.Server  // Handle new slugs not in build
  },

  // SERVER — HTML generated on EACH REQUEST at runtime
  // Use for: dynamic pages with real-time data, personalized pages
  { path: 'products',       renderMode: RenderMode.Server },
  { path: 'products/:id',   renderMode: RenderMode.Server },
  { path: 'search',         renderMode: RenderMode.Server },

  // CLIENT — CSR only, not rendered on server
  // Use for: authenticated/private pages, highly interactive UIs
  { path: 'dashboard',      renderMode: RenderMode.Client },
  { path: 'account/**',     renderMode: RenderMode.Client },
  { path: 'checkout',       renderMode: RenderMode.Client },
];
```

**Decision guide:**

| Route Type | Best Mode | Reason |
|---|---|---|
| Home / About / Pricing | `Prerender` | Static, SEO-critical, serve from CDN |
| Blog post / Docs page | `Prerender` + Server fallback | Mostly static, new content SSR'd |
| Product listing | `Server` | Real-time inventory, faceted filters |
| Auth protected pages | `Client` | Never expose user data in SSR |
| Admin dashboard | `Client` | Dynamic, no SEO benefit |

---

## Q20. What are `isPlatformBrowser()` and `isPlatformServer()`? Why are they needed?

**Answer:**
In an Angular SSR app, the same TypeScript code runs in **two environments**: the Node.js server and the browser. Some APIs (`localStorage`, `window`, `document`, `navigator`) exist only in the browser and will crash on the server.

`isPlatformBrowser()` and `isPlatformServer()` let you **guard platform-specific code**.

```typescript
import { Component, inject, PLATFORM_ID, OnInit } from '@angular/core';
import { isPlatformBrowser, isPlatformServer } from '@angular/common';

@Component({ selector: 'app-root', standalone: true, template: `...` })
export class AppComponent implements OnInit {
  private platformId = inject(PLATFORM_ID);

  ngOnInit() {
    if (isPlatformBrowser(this.platformId)) {
      // Browser-only APIs — safe here
      const savedTheme = localStorage.getItem('theme') ?? 'light';
      document.documentElement.setAttribute('data-theme', savedTheme);

      window.addEventListener('scroll', this.onScroll.bind(this));
    }

    if (isPlatformServer(this.platformId)) {
      // Server-only logic
      console.log('Running on server — skip browser APIs');
    }
  }

  onScroll() { /* ... */ }
}
```

**Service-level guard:**
```typescript
@Injectable({ providedIn: 'root' })
export class ThemeService {
  private platformId = inject(PLATFORM_ID);
  private isBrowser  = isPlatformBrowser(this.platformId);

  getTheme(): string {
    if (!this.isBrowser) return 'light';  // Default on server
    return localStorage.getItem('theme') ?? 'light';
  }

  setTheme(theme: string) {
    if (!this.isBrowser) return;
    localStorage.setItem('theme', theme);
    document.documentElement.setAttribute('data-theme', theme);
  }
}
```

**`afterNextRender` / `afterRender` (Angular 17+) — cleaner alternative:**
```typescript
import { afterNextRender, afterRender } from '@angular/core';

@Component({ ... })
export class ChartComponent {
  constructor() {
    // Only runs in the browser, after first render — no platform check needed
    afterNextRender(() => {
      this.initializeChart();  // Safe to access DOM
    });
  }
}
```

> **Senior Trap:** "Can I use `document.querySelector()` in `ngOnInit()` in an SSR app?"
> No — `ngOnInit()` runs on the server too. Use `afterNextRender()` (Angular 17+) or guard with `isPlatformBrowser()`.

# Category 5 — Advanced Concepts

---

## Q21. What is the Angular CDK? What are its key modules?

**Answer:**
**Angular CDK (Component Dev Kit)** is a library of unstyled, behavior-only primitives that provide the **interaction patterns** needed to build custom components — without imposing any visual design.

```bash
npm install @angular/cdk
```

**Key CDK modules:**

**1. Drag and Drop:**
```typescript
import { DragDropModule } from '@angular/cdk/drag-drop';

@Component({
  imports: [DragDropModule],
  template: `
    <div cdkDropList (cdkDropListDropped)="drop($event)">
      @for (item of items; track item) {
        <div cdkDrag>{{ item }}</div>
      }
    </div>
  `
})
export class KanbanComponent {
  items = ['Task 1', 'Task 2', 'Task 3'];

  drop(event: CdkDragDrop<string[]>) {
    moveItemInArray(this.items, event.previousIndex, event.currentIndex);
  }
}
```

**2. Overlay (for dropdowns, tooltips, popups):**
```typescript
import { Overlay, OverlayRef } from '@angular/cdk/overlay';
import { ComponentPortal } from '@angular/cdk/portal';

@Component({ ... })
export class DropdownTriggerComponent {
  private overlay = inject(Overlay);
  private overlayRef!: OverlayRef;

  open() {
    const positionStrategy = this.overlay.position()
      .flexibleConnectedTo(this.elementRef)
      .withPositions([{ originX: 'start', originY: 'bottom', overlayX: 'start', overlayY: 'top' }]);

    this.overlayRef = this.overlay.create({ positionStrategy, hasBackdrop: true });
    this.overlayRef.attach(new ComponentPortal(DropdownMenuComponent));
    this.overlayRef.backdropClick().subscribe(() => this.overlayRef.detach());
  }
}
```

**3. Virtual Scrolling (for large lists):**
```html
<cdk-virtual-scroll-viewport itemSize="48" style="height: 400px;">
  @for (item of largeDataset; track item.id) {
    <div *cdkVirtualFor="let item of largeDataset">{{ item.name }}</div>
  }
</cdk-virtual-scroll-viewport>
```

**4. Focus Management:**
```typescript
import { FocusTrap, FocusTrapFactory } from '@angular/cdk/a11y';

// Trap focus inside modal dialog (keyboard navigation)
const focusTrap: FocusTrap = this.focusTrapFactory.create(this.dialogEl.nativeElement);
focusTrap.focusInitialElement();
```

**5. Clipboard:**
```typescript
import { Clipboard } from '@angular/cdk/clipboard';

@Component({
  template: `<button (click)="copy()">Copy</button>`
})
export class CopyButtonComponent {
  private clipboard = inject(Clipboard);

  copy() { this.clipboard.copy('https://angular.dev'); }
}
```

**CDK modules overview:**

| Module | Purpose |
|---|---|
| `drag-drop` | Sortable lists, kanban boards |
| `overlay` | Popups, dropdowns, tooltips |
| `portal` | Render components/templates into DOM portals |
| `virtual-scroll` | Render only visible rows (virtualization) |
| `a11y` | Focus trap, live announcer, keyboard manager |
| `layout` | Media query breakpoint detection |
| `clipboard` | Copy-to-clipboard |
| `stepper` | Step-by-step wizards |
| `table` | Data table foundation |

---

## Q22. Explain content projection with `<ng-content>`. What is multi-slot projection?

**Answer:**
**Content projection** allows a parent component to inject template content into a specific location inside a child component — similar to Vue's `<slot>` or React's `children`.

**Single-slot projection:**
```typescript
// card.component.ts
@Component({
  selector: 'app-card',
  standalone: true,
  template: `
    <div class="card">
      <ng-content />   <!-- projected content goes here -->
    </div>
  `
})
export class CardComponent {}
```
```html
<!-- Usage -->
<app-card>
  <h2>Card Title</h2>
  <p>Card body content here.</p>
</app-card>
```

**Multi-slot projection with `select`:**
```typescript
// dialog.component.ts
@Component({
  selector: 'app-dialog',
  standalone: true,
  template: `
    <div class="dialog">
      <header class="dialog-header">
        <ng-content select="[dialog-title]" />  <!-- Matches elements with attribute -->
      </header>
      <main class="dialog-body">
        <ng-content select=".dialog-body" />    <!-- Matches by CSS class -->
      </main>
      <footer class="dialog-footer">
        <ng-content select="app-dialog-actions" /> <!-- Matches by element name -->
      </footer>
      <ng-content />  <!-- Catches everything else (default slot) -->
    </div>
  `
})
export class DialogComponent {}
```
```html
<!-- Usage -->
<app-dialog>
  <h1 dialog-title>Confirm Delete</h1>
  <p class="dialog-body">This action cannot be undone.</p>
  <app-dialog-actions>
    <button (click)="cancel()">Cancel</button>
    <button (click)="confirm()">Delete</button>
  </app-dialog-actions>
</app-dialog>
```

**`ngProjectAs` — project as a different selector:**
```html
<!-- ng-container is transparent in DOM but projects as a named slot -->
<ng-container ngProjectAs="[dialog-title]">
  <h1>My Title</h1>
</ng-container>
```

**Accessing projected content with `@ContentChild`:**
```typescript
@Component({ ... })
export class TabGroupComponent {
  @ContentChildren(TabComponent) tabs!: QueryList<TabComponent>;

  ngAfterContentInit() {
    // tabs.changes is an Observable that fires when projected content changes
    this.tabs.changes.subscribe(() => this.updateActiveTab());
  }
}
```

---

## Q23. Explain `@HostBinding` and `@HostListener`. When would you use them?

**Answer:**
`@HostBinding` and `@HostListener` allow a component or directive to **manipulate its host element** directly — the element it is applied to.

**`@HostBinding` — bind property/attribute/class/style on host:**
```typescript
@Directive({
  selector: '[appHighlight]',
  standalone: true,
})
export class HighlightDirective {
  @Input() highlightColor = '#ffff00';

  // Bind host element's style.backgroundColor
  @HostBinding('style.backgroundColor') bgColor = '';

  // Add CSS class conditionally
  @HostBinding('class.is-highlighted') isHighlighted = false;

  // Bind aria attribute
  @HostBinding('attr.aria-label') ariaLabel = 'Highlighted item';

  @HostListener('mouseenter')
  onMouseEnter() {
    this.bgColor = this.highlightColor;
    this.isHighlighted = true;
  }

  @HostListener('mouseleave')
  onMouseLeave() {
    this.bgColor = '';
    this.isHighlighted = false;
  }
}
```
```html
<p appHighlight highlightColor="#ffd700">Hover over me</p>
```

**`@HostListener` — listen to events on the host or global targets:**
```typescript
@Directive({ selector: '[appKeyboardShortcut]', standalone: true })
export class KeyboardShortcutDirective {
  @Input() shortcutKey = 's';
  private save = output<void>();

  // Listen to document-level keydown (not just host element)
  @HostListener('document:keydown', ['$event'])
  onKeydown(event: KeyboardEvent) {
    if ((event.ctrlKey || event.metaKey) && event.key === this.shortcutKey) {
      event.preventDefault();
      this.save.emit();
    }
  }
}
```

**Modern alternative — `host` metadata property (preferred in Angular 15+):**
```typescript
@Directive({
  selector: '[appHighlight]',
  standalone: true,
  host: {
    '(mouseenter)':            'onMouseEnter()',
    '(mouseleave)':            'onMouseLeave()',
    '[style.background-color]': 'bgColor',
    '[class.is-highlighted]':  'isHighlighted',
    '[attr.aria-pressed]':     'isHighlighted',
  }
})
export class HighlightDirective { ... }
```
The `host` property is more performant (no decorators processed at runtime) and tree-shakeable.

---

## Q24. What is `InjectionToken`? When do you use it?

**Answer:**
**`InjectionToken`** creates a typed token for injecting non-class values into Angular's DI system — primitives, configuration objects, factory functions, or abstract interfaces.

**Why not just use a string token?**
String tokens have no type safety and can collide between libraries. `InjectionToken` is **typed** and **unique by identity**.

**Configuration object:**
```typescript
// app.config-token.ts
export interface AppConfig {
  apiUrl:     string;
  apiTimeout: number;
  featureFlags: {
    darkMode:   boolean;
    betaEditor: boolean;
  };
}

export const APP_CONFIG = new InjectionToken<AppConfig>('APP_CONFIG', {
  // Default factory — optional
  providedIn: 'root',
  factory: () => ({
    apiUrl:     'https://api.example.com',
    apiTimeout: 5000,
    featureFlags: { darkMode: false, betaEditor: false },
  })
});
```

```typescript
// app.config.ts — override per environment
export const appConfig: ApplicationConfig = {
  providers: [
    {
      provide: APP_CONFIG,
      useValue: {
        apiUrl: environment.apiUrl,
        apiTimeout: 8000,
        featureFlags: { darkMode: true, betaEditor: false },
      }
    },
  ]
};
```

```typescript
// feature.service.ts — consume the token
@Injectable({ providedIn: 'root' })
export class FeatureService {
  private config = inject(APP_CONFIG);

  get apiUrl() { return this.config.apiUrl; }

  isDarkMode() { return this.config.featureFlags.darkMode; }
}
```

**Abstract interface token (for swappable implementations):**
```typescript
export abstract class LoggerService {
  abstract log(message: string): void;
  abstract error(message: string, err?: Error): void;
}

export const LOGGER_TOKEN = new InjectionToken<LoggerService>('LOGGER');

// Provide different loggers per environment
{ provide: LOGGER_TOKEN, useClass: isDevMode() ? ConsoleLogger : SentryLogger }

// Consume
private logger = inject(LOGGER_TOKEN);
```

---

## Q25. Explain `forRoot()` vs `forChild()`. Why does this pattern exist?

**Answer:**
The `forRoot()` / `forChild()` pattern is a **module-level DI pattern** that ensures a service is provided as a **singleton at the root level**, while still allowing modules to configure it.

**The problem it solves:**
If `SharedModule` declares a service with `providers: [AuthService]`, and two lazy-loaded modules both import `SharedModule`, each creates its **own instance** of `AuthService` — breaking singleton behavior.

**`forRoot()` — call ONCE in AppModule (singleton):**
```typescript
// logger.module.ts
@NgModule({ ... })
export class LoggerModule {
  static forRoot(config: LoggerConfig): ModuleWithProviders<LoggerModule> {
    return {
      ngModule: LoggerModule,
      providers: [
        LoggerService,  // ← provided ONCE at root
        { provide: LOGGER_CONFIG, useValue: config }
      ]
    };
  }

  // forChild — import in feature modules (components/pipes, NO providers)
  static forChild(): ModuleWithProviders<LoggerModule> {
    return {
      ngModule: LoggerModule,
      providers: []  // ← no service providers here
    };
  }
}
```

```typescript
// app.module.ts
@NgModule({
  imports: [
    LoggerModule.forRoot({ level: 'info', endpoint: '/api/logs' })
  ]
})
export class AppModule {}

// feature.module.ts
@NgModule({
  imports: [
    LoggerModule.forChild()  // Gets components/directives, shares root singleton service
  ]
})
export class FeatureModule {}
```

**Real-world examples:**
```typescript
RouterModule.forRoot(routes)       // ← in AppModule (singleton Router service)
RouterModule.forChild(featureRoutes) // ← in FeatureModule
```

**In standalone apps — this pattern is replaced by:**
```typescript
// app.config.ts
provideRouter(routes)  // ← always singleton
// Feature modules: import RouterModule in component, no forChild needed
```

> **Trap:** "Is `forRoot()` needed in standalone Angular apps?"
> No — standalone apps use `provide*()` functions which are inherently singleton-aware via `providedIn: 'root'`. `forRoot/forChild` is an NgModule-era pattern.

# Category 6 — Best Practices & Architecture

---

## Q26. What is the Smart/Dumb (Container/Presentational) Component Pattern?

**Answer:**
The **Smart/Dumb** (also called **Container/Presentational**) pattern separates components by their responsibility:

| | Smart Component | Dumb Component |
|---|---|---|
| Also called | Container, Connected | Presentational, Pure |
| Knows about | Store, services, routing | Nothing outside itself |
| Data source | Injects services/store | Receives via `@Input()` |
| Events | Handles business logic | Emits via `@Output()` |
| Change detection | Default (or OnPush with signals) | **Always OnPush** |
| Reusability | Low — context-specific | **High — reusable anywhere** |
| Testability | Harder (needs service mocks) | Easy (pure I/O testing) |

**Smart component (container):**
```typescript
// product-list-page.component.ts — Smart
@Component({
  selector: 'app-product-list-page',
  standalone: true,
  imports: [ProductCardComponent, ProductFiltersComponent, AsyncPipe],
  template: `
    <app-product-filters
      [categories]="categories$ | async"
      (filterChanged)="onFilterChanged($event)" />

    @for (product of filteredProducts$ | async; track product.id) {
      <app-product-card
        [product]="product"
        [isFavorite]="(favorites$ | async)?.includes(product.id)"
        (addToCart)="onAddToCart($event)"
        (toggleFavorite)="onToggleFavorite($event)" />
    }
  `,
})
export class ProductListPageComponent {
  private store          = inject(ProductStore);
  private cartService    = inject(CartService);
  private favoriteService = inject(FavoriteService);

  categories$      = this.store.categories$;
  filteredProducts$ = this.store.filteredProducts$;
  favorites$       = this.favoriteService.favorites$;

  onFilterChanged(filter: ProductFilter) { this.store.setFilter(filter); }
  onAddToCart(product: Product)          { this.cartService.addItem(product); }
  onToggleFavorite(id: string)           { this.favoriteService.toggle(id); }
}
```

**Dumb component (presentational):**
```typescript
// product-card.component.ts — Dumb
@Component({
  selector: 'app-product-card',
  standalone: true,
  changeDetection: ChangeDetectionStrategy.OnPush,  // Always OnPush
  template: `
    <div class="card" [class.favorite]="isFavorite">
      <img [src]="product.imageUrl" [alt]="product.name" loading="lazy">
      <h3>{{ product.name }}</h3>
      <p class="price">{{ product.price | currency }}</p>
      <button (click)="addToCart.emit(product)">Add to Cart</button>
      <button (click)="toggleFavorite.emit(product.id)">
        {{ isFavorite ? '❤️' : '🤍' }}
      </button>
    </div>
  `,
})
export class ProductCardComponent {
  // Pure I/O — no service injection
  product      = input.required<Product>();
  isFavorite   = input(false);
  addToCart    = output<Product>();
  toggleFavorite = output<string>();
}
```

> **Benefit:** Dumb components can be tested by simply providing `@Input()` values and asserting `@Output()` emissions — no mocking required.

---

## Q27. What is a recommended Angular feature folder structure?

**Answer:**
Angular's style guide and community best practices converge on a **feature-based** folder structure:

```
src/
├── app/
│   ├── core/                       ← App-wide singletons (loaded once)
│   │   ├── guards/
│   │   │   ├── auth.guard.ts
│   │   │   └── role.guard.ts
│   │   ├── interceptors/
│   │   │   ├── auth.interceptor.ts
│   │   │   └── error.interceptor.ts
│   │   ├── services/
│   │   │   ├── auth.service.ts
│   │   │   └── logger.service.ts
│   │   └── models/
│   │       └── user.model.ts
│   │
│   ├── shared/                     ← Reusable components, directives, pipes
│   │   ├── components/
│   │   │   ├── button/
│   │   │   ├── card/
│   │   │   └── modal/
│   │   ├── directives/
│   │   │   └── highlight.directive.ts
│   │   ├── pipes/
│   │   │   └── truncate.pipe.ts
│   │   └── index.ts                ← Barrel export
│   │
│   ├── features/                   ← Feature areas (lazy loaded)
│   │   ├── products/
│   │   │   ├── components/
│   │   │   │   ├── product-card/
│   │   │   │   └── product-filters/
│   │   │   ├── pages/
│   │   │   │   ├── product-list/
│   │   │   │   └── product-detail/
│   │   │   ├── services/
│   │   │   │   └── product.service.ts
│   │   │   ├── store/
│   │   │   │   ├── product.store.ts
│   │   │   │   └── product.selectors.ts
│   │   │   ├── models/
│   │   │   │   └── product.model.ts
│   │   │   └── products.routes.ts
│   │   │
│   │   ├── cart/
│   │   └── checkout/
│   │
│   ├── app.component.ts
│   ├── app.config.ts
│   └── app.routes.ts
│
├── environments/
│   ├── environment.ts              ← Development values
│   └── environment.prod.ts        ← Production values
└── main.ts
```

**Barrel exports (`index.ts`):**
```typescript
// shared/index.ts
export * from './components/button/button.component';
export * from './components/card/card.component';
export * from './directives/highlight.directive';
export * from './pipes/truncate.pipe';
```
```typescript
// Usage — clean import from barrel
import { ButtonComponent, CardComponent } from '@app/shared';
// Instead of: import { ButtonComponent } from '../../../shared/components/button/button.component'
```

**Path aliases in `tsconfig.json`:**
```json
{
  "compilerOptions": {
    "paths": {
      "@app/core/*":    ["src/app/core/*"],
      "@app/shared":    ["src/app/shared/index.ts"],
      "@app/features/*": ["src/app/features/*"]
    }
  }
}
```

---

## Q28. How do you manage environment configuration in Angular?

**Answer:**

**Built-in environment files:**
```typescript
// src/environments/environment.ts (development)
export const environment = {
  production:  false,
  apiUrl:      'http://localhost:3000/api',
  featureFlags: {
    betaEditor: true,
    darkMode:   false,
  }
};

// src/environments/environment.prod.ts (production)
export const environment = {
  production:  true,
  apiUrl:      'https://api.myapp.com',
  featureFlags: {
    betaEditor: false,
    darkMode:   false,
  }
};
```

**`angular.json` — file replacement at build:**
```json
{
  "configurations": {
    "production": {
      "fileReplacements": [{
        "replace": "src/environments/environment.ts",
        "with":    "src/environments/environment.prod.ts"
      }]
    },
    "staging": {
      "fileReplacements": [{
        "replace": "src/environments/environment.ts",
        "with":    "src/environments/environment.staging.ts"
      }]
    }
  }
}
```

```bash
ng build --configuration production  # Uses environment.prod.ts
ng build --configuration staging     # Uses environment.staging.ts
ng serve                             # Uses environment.ts (dev)
```

**Better pattern — Inject via `InjectionToken` (testable, DI-friendly):**
```typescript
// app.config.ts — provide via DI instead of direct import
import { APP_CONFIG } from './core/tokens/app-config.token';
import { environment } from '../environments/environment';

export const appConfig: ApplicationConfig = {
  providers: [
    { provide: APP_CONFIG, useValue: environment },
  ]
};
```
```typescript
// Any service or component
private config = inject(APP_CONFIG);
get apiUrl() { return this.config.apiUrl; }
```

**Why prefer DI over direct `environment` imports?**
- Direct import: hard to mock in tests
- DI token: easy to override in `TestBed.configureTestingModule({ providers: [...] })`

---

## Q29. What are Angular's built-in security protections? How do you prevent XSS?

**Answer:**
Angular has several **built-in** security mechanisms that protect against the OWASP Top 10 vulnerabilities.

**1. Automatic XSS prevention (default — template binding):**
```typescript
// Angular AUTOMATICALLY sanitizes/escapes all interpolated & bound values
this.userInput = '<script>alert("XSS")</script>';
```
```html
<!-- Safe — Angular HTML-encodes this output -->
<p>{{ userInput }}</p>

<!-- Safe — Angular sanitizes [innerHTML] by stripping dangerous tags/attrs -->
<div [innerHTML]="userInput"></div>  // <script> stripped automatically
```

**2. `DomSanitizer` — only bypass sanitization when you ABSOLUTELY trust the source:**
```typescript
import { DomSanitizer, SafeHtml, SafeUrl } from '@angular/platform-browser';

@Component({ ... })
export class EmbedComponent {
  private sanitizer = inject(DomSanitizer);

  // Only use bypassSecurity* for trusted, controlled content
  getSafeHtml(html: string): SafeHtml {
    // WARNING: ONLY if html comes from your own controlled CMS, never from user input
    return this.sanitizer.bypassSecurityTrustHtml(html);
  }

  getSafeUrl(url: string): SafeUrl {
    return this.sanitizer.bypassSecurityTrustUrl(url);
  }
}
```

**Safe methods:**
```typescript
sanitizer.bypassSecurityTrustHtml(html)       // Trust raw HTML
sanitizer.bypassSecurityTrustStyle(style)     // Trust CSS
sanitizer.bypassSecurityTrustUrl(url)         // Trust URL (href, src)
sanitizer.bypassSecurityTrustResourceUrl(url) // Trust URL for iframes/scripts
sanitizer.bypassSecurityTrustScript(script)   // Trust JavaScript
```

**3. HTTP Security:**
```typescript
// CSRF — Angular's HttpClient does NOT auto-add CSRF tokens
// Configure your API server to set a cookie, then read it client-side:
provideHttpClient(
  withXsrfConfiguration({
    cookieName: 'XSRF-TOKEN',
    headerName: 'X-XSRF-TOKEN',
  })
)

// Content Security Policy — set via server headers (not Angular's job)
// Content-Security-Policy: default-src 'self'; script-src 'self'
```

**4. Route guards — authorization:**
```typescript
export const authGuard: CanActivateFn = (route, state) => {
  const auth   = inject(AuthService);
  const router = inject(Router);
  return auth.isLoggedIn()
    ? true
    : router.createUrlTree(['/login'], { queryParams: { returnUrl: state.url } });
};
```

**OWASP Top 10 — Angular relevance:**

| OWASP | Risk | Angular's Defense |
|---|---|---|
| A01 Broken Access Control | Route accessed without auth | Route guards + backend authorization |
| A03 Injection (XSS) | Script injection via user input | Automatic HTML sanitization |
| A05 Security Misconfiguration | Open CORS, no CSP | Server-side headers, not Angular |
| A07 Authentication Failures | Token theft | HttpOnly cookies, short-lived JWTs |
| A08 Software/Data Integrity | Malicious `[innerHTML]` | Never `bypassSecurity` user input |

> **Senior Trap:** "Is Angular's sanitization enough by itself?"
> For the frontend, yes. But backend validation and authorization are equally critical — never trust client-side guards alone.

---

## Q30. What is tree-shaking? How does Angular support it?

**Answer:**
**Tree-shaking** is the process where the JavaScript bundler (webpack/esbuild) statically analyzes imports and **removes code that is never used** (dead code) from the final bundle.

**Why it matters:**
```bash
# Without tree-shaking: entire library included
# With tree-shaking: only imported functions included

import { Observable, Subject, BehaviorSubject, from, of } from 'rxjs';
# If you only ever use 'of' and 'from', the rest of rxjs is dropped
```

**How Angular enables tree-shaking:**

**1. `providedIn: 'root'` (services):**
```typescript
// NOT tree-shakeable — always included if LoggerModule is imported
@NgModule({ providers: [LoggerService] })
export class LoggerModule {}

// Tree-shakeable — only included if actually injected somewhere
@Injectable({ providedIn: 'root' })
export class LoggerService {}
// If nothing injects LoggerService, it's excluded from the bundle
```

**2. Standalone components — granular imports:**
```typescript
// Standalone — only components actually used in template are bundled
@Component({
  imports: [
    NgIf,       // ← included (used in template)
    NgFor,      // ← included
    // NgSwitch  ← NOT imported, so excluded from bundle
  ],
  template: `<div *ngIf="..."><span *ngFor="...">...</span></div>`
})
export class MyComponent {}
```

**3. `sideEffects: false` in `package.json`:**
```json
// library/package.json
{
  "name": "my-angular-lib",
  "sideEffects": false  // ← tells bundler: safe to tree-shake anything unused
}
```

**4. Angular standalone APIs vs NgModule:**
```typescript
// NgModule approach — entire RouterModule always included
imports: [RouterModule.forRoot(routes)]

// Standalone approach — only router features you use are bundled
provideRouter(routes, withPreloading(PreloadAllModules))  // exact features
```

**Bundle analysis:**
```bash
ng build --stats-json
npx webpack-bundle-analyzer dist/my-app/browser/stats.json
# OR
source-map-explorer dist/my-app/browser/main.*.js
```

**Tree-shaking checklist:**
- ✅ Services use `providedIn: 'root'`
- ✅ Use standalone components with granular imports
- ✅ Import from specific sub-packages: `import { map } from 'rxjs/operators'`
- ✅ Use `provideRouter()` + `with*()` functions instead of `RouterModule`
- ✅ Enable Angular's production build: `ng build --configuration production`
- ❌ Avoid wildcard imports: `import * as _ from 'lodash'`
- ❌ Avoid `NgModule` providers array for singleton services

# Quick Reference Cheatsheet — Set 2

---

## Signals API Quick Reference

| API | Usage | Notes |
|---|---|---|
| `signal(val)` | `count = signal(0)` | Writable signal |
| `computed(() => ...)` | `double = computed(() => count() * 2)` | Read-only derived signal |
| `effect(() => ...)` | `effect(() => console.log(count()))` | Side effect on signal change |
| `toSignal(obs$)` | `data = toSignal(http.get(...))` | Observable → Signal |
| `toObservable(sig)` | `count$ = toObservable(count)` | Signal → Observable |
| `input()` | `name = input('')` | Signal-based `@Input` |
| `input.required()` | `id = input.required<string>()` | Required signal input |
| `output()` | `clicked = output<void>()` | Signal-based `@Output` |
| `linkedSignal()` | `linkedSignal(() => source())` | Writable derived signal |
| `resource()` | `resource({ loader: () => fetch(...) })` | Async resource signal |
| `httpResource()` | `httpResource('/api/products')` | HTTP-backed resource signal |

---

## NgRx Core Concepts

```typescript
// Action
const load = createAction('[Products] Load');
const loadSuccess = createAction('[Products] Load Success', props<{ products: Product[] }>());

// Reducer
const reducer = createReducer(
  initialState,
  on(load, state => ({ ...state, loading: true })),
  on(loadSuccess, (state, { products }) => ({ ...state, products, loading: false })),
);

// Selector
const selectProducts = createSelector(
  createFeatureSelector<State>('products'),
  state => state.products
);

// Effect
loadProducts$ = createEffect(() => this.actions$.pipe(
  ofType(load),
  switchMap(() => this.service.getAll().pipe(
    map(products => loadSuccess({ products })),
    catchError(err => of(loadFailure({ error: err.message })))
  ))
));

// Component
products$ = this.store.select(selectProducts);
this.store.dispatch(load());
```

---

## NgRx Signals Store vs NgRx (Redux) Comparison

| Feature | NgRx Redux | NgRx Signals Store |
|---|---|---|
| Files needed | 4+ (actions, reducer, effects, selectors) | 1 (signalStore) |
| State access | `store.select(selector)` → Observable | Direct signal: `store.products()` |
| Update | `store.dispatch(action)` | `patchState(store, {...})` |
| Side effects | `createEffect` + `ofType` | `rxMethod` |
| DevTools | Full Redux DevTools | Basic |
| Bundle size | ~25KB | ~15KB |

---

## Testing Cheatsheet

```typescript
// Setup
await TestBed.configureTestingModule({ imports: [StandaloneComp] }).compileComponents();
fixture = TestBed.createComponent(MyComponent);
component = fixture.componentInstance;
fixture.detectChanges();

// Query DOM
fixture.debugElement.query(By.css('.btn')).nativeElement.click();
fixture.debugElement.queryAll(By.directive(MyDirective));

// Mock service
const spy = jasmine.createSpyObj('AuthService', ['isLoggedIn', 'login']);
spy.isLoggedIn.and.returnValue(true);
{ provide: AuthService, useValue: spy }

// spyOn
spyOn(router, 'navigate').and.resolveTo(true);
expect(router.navigate).toHaveBeenCalledWith(['/home']);

// HTTP testing
httpMock = TestBed.inject(HttpTestingController);
const req = httpMock.expectOne('/api/products');
expect(req.request.method).toBe('GET');
req.flush([{ id: 1, name: 'Laptop' }]);
httpMock.verify();

// Guard testing
const result = TestBed.runInInjectionContext(() => authGuard({} as any, {} as any));
expect(result).toBeTrue();

// fakeAsync
fakeAsync(() => { component.search('a'); tick(300); fixture.detectChanges(); })
```

---

## SSR & Hydration Commands

```bash
# Add SSR to existing project
ng add @angular/ssr

# Build SSR project
ng build

# Run SSR server
node dist/my-app/server/server.mjs

# Serve with SSR in development
ng serve  # (with SSR configured, serves with hydration)
```

| Feature | Config | Angular Version |
|---|---|---|
| Non-Destructive Hydration | `provideClientHydration()` | Stable ≥ 17 |
| HTTP Transfer Cache | `withHttpTransferCache()` (default) | Stable ≥ 17 |
| Incremental Hydration | `withIncrementalHydration()` | Dev Preview 18, Stable 19 |
| Route-level Render Mode | `app.routes.server.ts` + `RenderMode.*` | Dev Preview 18, Stable 19 |

---

## Angular CDK Quick Reference

| Module | Import | Key Directive/Class |
|---|---|---|
| Drag & Drop | `DragDropModule` | `cdkDrag`, `cdkDropList`, `moveItemInArray()` |
| Virtual Scroll | `ScrollingModule` | `cdk-virtual-scroll-viewport`, `*cdkVirtualFor` |
| Overlay | `OverlayModule` | `Overlay`, `OverlayRef`, `ComponentPortal` |
| Focus Trap | `A11yModule` | `FocusTrap`, `cdkTrapFocus` |
| Clipboard | `ClipboardModule` | `Clipboard.copy(text)` |
| Breakpoints | `LayoutModule` | `BreakpointObserver.observe([Breakpoints.Handset])` |

---

## Set 2 — 30 Question Index

| # | Question | Category |
|---|---|---|
| Q1 | Signal graph internals — dependency tracking | Signals |
| Q2 | `toSignal()` / `toObservable()` — RxJS interop | Signals |
| Q3 | `input()`, `output()`, signal I/O | Signals |
| Q4 | `linkedSignal()`, `resource()`, `httpResource()` | Signals |
| Q5 | Signals vs RxJS decision matrix | Signals |
| Q6 | NgRx — Store, Actions, Reducers, Selectors, Effects | State Mgmt |
| Q7 | Using NgRx in components | State Mgmt |
| Q8 | NgRx Signals Store (`@ngrx/signals`) | State Mgmt |
| Q9 | State management alternatives (Services+Signals, NGXS) | State Mgmt |
| Q10 | Subject vs BehaviorSubject vs ReplaySubject | State Mgmt |
| Q11 | Unit testing components with `TestBed` | Testing |
| Q12 | Testing Angular services | Testing |
| Q13 | `HttpClientTestingModule` + `HttpTestingController` | Testing |
| Q14 | Mocking — `createSpyObj`, `spyOn`, stubs | Testing |
| Q15 | Testing routes and guards | Testing |
| Q16 | SSR setup with `@angular/ssr` | SSR |
| Q17 | Non-Destructive Hydration vs old SSR | SSR |
| Q18 | Incremental Hydration | SSR |
| Q19 | Route-level Render Mode (`Prerender`/`Server`/`Client`) | SSR |
| Q20 | `isPlatformBrowser()`, `afterNextRender()` | SSR |
| Q21 | Angular CDK — key modules | Advanced |
| Q22 | Content projection — single/multi-slot, `ngProjectAs` | Advanced |
| Q23 | `@HostBinding`, `@HostListener`, `host` metadata | Advanced |
| Q24 | `InjectionToken` — typed DI for primitives | Advanced |
| Q25 | `forRoot()` vs `forChild()` | Advanced |
| Q26 | Smart/Dumb component pattern | Best Practices |
| Q27 | Feature folder structure, barrel exports, path aliases | Best Practices |
| Q28 | Environment config — `environment.ts`, DI token pattern | Best Practices |
| Q29 | Angular security — XSS, `DomSanitizer`, CSRF, OWASP | Best Practices |
| Q30 | Tree-shaking — `providedIn: 'root'`, standalone imports | Best Practices |

---

## Set 3 Preview — Coming Next

| Category | Topics |
|---|---|
| Performance Optimization | `OnPush`, `trackBy`, lazy loading, `NgOptimizedImage`, bundle budgets |
| Angular Animations | `trigger`, `state`, `transition`, `animate`, route transitions |
| Internationalization (i18n) | `$localize`, `@angular/localize`, locale IDs, `DatePipe` locales |
| Micro-frontend / Module Federation | Webpack Module Federation, shared dependencies, Angular Elements |
| Monorepos / Nx | Nx workspace, `nx affected`, libs vs apps, dependency graph |
| Angular DevTools | Profiler, component explorer, signal dependency graph |